In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
import re

In [0]:
def cust_dim(df):
  customer_df=df.select('Customer_Id','Customer_Name','Address')
  df=df.drop('Customer_Name','Address') #dropping columns from main table which are taken in cusomer df
  customer_df=customer_df.dropDuplicates(['Customer_Id'])
  # customer_df.show()
  return customer_df, df

def plant_dim(df):
  plant_df=df.select('Plant_Id','Sorg','Dch','Div')
  #dropping columns from main table which are taken in pant df
  df=df.drop('Sorg','Dch','Div')
  plant_df=plant_df.dropDuplicates(['Plant_Id','Dch'])
  return plant_df, df

def custgrp_dim(df):
  custgrp_df=df.select('Customer_Grp_Id','Customer_Grp_Name')
#   print("Before",custgrp_df.count())
  #dropping columns from main table which are taken in pant df
  df=df.drop('Customer_Grp_Name')
  custgrp_df=custgrp_df.dropDuplicates(['Customer_Grp_Id'])
#   print("After",custgrp_df.count())
  return custgrp_df, df

def product_dim(df):
  material_df=df.select('Product_Id','Product_Name','Product_Category','Product_Type','Andris','EWH_Groups')
#   print("Before",material_df.count())
  #dropping columns from main table which are taken in pant df
  df=df.drop('Product_Name','Product_Category','Product_Type','Andris','EWH_Groups')
  material_df=material_df.dropDuplicates(['Product_Id'])
#   print("After",material_df.count())
  return material_df, df

def location_dim(df):
  location_df=df.select(col('State_Name').alias('StateName'),col('Sales_Zone').alias('SalesZone'),col('Sales_Region').alias('SalesRegion'))
  location_df=location_df.dropDuplicates(['StateName','SalesZone','SalesRegion'])
  # Add a unique id column to the existing dataframe
  location_df = location_df.withColumn("StateId", monotonically_increasing_id())
  # Convert the id column to a string
  location_df = location_df.withColumn("StateId", concat(lit("S"), col("StateId").cast("string")))
  return location_df, df

In [0]:
from datetime import datetime, timedelta
# Define a threshold for recent modification time (e.g., within the last 24 hours)
threshold_time = datetime.now() - timedelta(hours=72)
for filePath in dbutils.fs.ls('mnt/silver'):
    if filePath.name.endswith('.parquet') and datetime.fromtimestamp(filePath.modificationTime / 1000) > threshold_time:
        path=filePath[0]
        print(path)
        df = spark.read.parquet(path)
        # # df.show()
        # print(df.count())
        # # df.printSchema()
        #Creating Customer Dimention
        customer_df,df=cust_dim(df)
        #Creating Plant Dimention
        plant_df,df=plant_dim(df)
        #Creating Customer_group Dimention
        custgrp_df,df=custgrp_dim(df)
        #Creating product Dimention
        product_df,df=product_dim(df)
        #Creating Location Dimention
        location_df,df=location_dim(df)
        location_df.dropDuplicates(['StateId'])
        #addting loaction_id to fact data frame 
        df = df.join(location_df,
                            (df.State_Name == location_df.StateName) &
                            (df.Sales_Zone == location_df.SalesZone) &
                            (df.Sales_Region == location_df.SalesRegion),
                            "left")
        df=df.drop("State_Name","Sales_Zone","Sales_Region","StateName","SalesZone","SalesRegion")

dbfs:/mnt/silver/part-00000-tid-4393133502302247746-7f49a790-f705-4c54-8f69-8ddd674456fc-338-1.c000.snappy.parquet


---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-3825900227141272>, line 13
      8 df = spark.read.parquet(path)
      9 # # df.show()
     10 # print(df.count())
     11 # # df.printSchema()
     12 #Creating Customer Dimention
---> 13 customer_df,df=cust_dim(df)
     14 #Creating Plant Dimention
     15 plant_df,df=plant_dim(df)

NameError: name 'cust_dim' is not defined

In [0]:
from datetime import datetime,timedelta
opt_path='mnt/gold'
# Assuming df is your DataFrame and opt_path is your output path
# Generate a timestamp to make the file path unique
timestamp = datetime.now()+timedelta(hours=5,minutes=30)
# .strftime("%Y%m%d%H%M")
# x=f"{timestamp.year}{timestamp.month}{timestamp.day}"
# print(x)

# Append the timestamp to the output path customer data
output_path_customerDim = f"{opt_path}/customerData/{timestamp.year}/{timestamp.month}/{timestamp.day}/customer_dim_{timestamp.hour}{timestamp.minute}{timestamp.second}.csv"

# Write DataFrame to the output path with timestamp
customer_df.write.option("header", "true").csv(output_path_customerDim, mode='append')


# Append the timestamp to the output path to product data
output_path_productDim = f"{opt_path}/productData/{timestamp.year}/{timestamp.month}/{timestamp.day}/product_dim_{timestamp.hour}{timestamp.minute}{timestamp.second}.csv"

# Write DataFrame to the output path with timestamp
product_df.write.option("header", "true").csv(output_path_productDim, mode='append')


# Append the timestamp to the output path to plant data
output_path_plantDim = f"{opt_path}/plantData/{timestamp.year}/{timestamp.month}/{timestamp.day}/plant_dim_{timestamp.hour}{timestamp.minute}{timestamp.second}.csv"

# Write DataFrame to the output path with timestamp
plant_df.write.option("header", "true").csv(output_path_plantDim, mode='append')

# Append the timestamp to the output path to customer grp
output_path_custgrpDim = f"{opt_path}/custgrpData/{timestamp.year}/{timestamp.month}/{timestamp.day}/custgrp_dim_{timestamp.hour}{timestamp.minute}{timestamp.second}.csv"

# Write DataFrame to the output path with timestamp 
custgrp_df.write.option("header", "true").csv(output_path_custgrpDim, mode='append')

# Append the timestamp to the output path of location df
output_path_loacationDim = f"{opt_path}/locationData/{timestamp.year}/{timestamp.month}/{timestamp.day}/location_dim_{timestamp.hour}{timestamp.minute}{timestamp.second}.csv"

# Write DataFrame to the output path with timestamp
location_df.write.option("header", "true").csv(output_path_loacationDim, mode='append')

# Append the timestamp to the output path of sales fact
output_path_salesFact = f"{opt_path}/salesfactData/{timestamp.year}/{timestamp.month}/{timestamp.day}/sales_fact_{timestamp.hour}{timestamp.minute}{timestamp.second}.csv"

# Write DataFrame to the output path with timestamp 
df.write.option("header", "true").csv(output_path_salesFact, mode='append')





---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-3825900227141273>, line 14
     11 output_path_customerDim = f"{opt_path}/customerData/{timestamp.year}/{timestamp.month}/{timestamp.day}/customer_dim_{timestamp.hour}{timestamp.minute}{timestamp.second}.csv"
     13 # Write DataFrame to the output path with timestamp
---> 14 customer_df.write.option("header", "true").csv(output_path_customerDim, mode='append')
     17 # Append the timestamp to the output path to product data
     18 output_path_productDim = f"{opt_path}/productData/{timestamp.year}/{timestamp.month}/{timestamp.day}/product_dim_{timestamp.hour}{timestamp.minute}{timestamp.second}.csv"

NameError: name 'customer_df' is not defined